In [ ]:
{
 "nbformat": 4,
 "nbformat_minor": 5,
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "name": "python",
   "version": "3.10.0"
  },
  "accelerator": "GPU",
  "colab": {
   "provenance": [],
   "gpuType": "T4"
  }
 },
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# GPU Neural Network Layer Accelerator\n",
    "\n",
    "This notebook compiles and runs CUDA kernels for matrix multiplication, ReLU, and Softmax on a free T4 GPU.\n",
    "\n",
    "**Before running:** Go to `Runtime → Change runtime type → T4 GPU`\n",
    "\n",
    "Then run all cells top to bottom with `Runtime → Run all`."
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 1 — Verify GPU"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "!nvidia-smi"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 2 — Clone repo"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "import os\n",
    "\n",
    "if not os.path.exists('gpu-neural-net-accelerator'):\n",
    "    !git clone https://github.com/BlissPhinehas/gpu-neural-net-accelerator.git\n",
    "\n",
    "%cd gpu-neural-net-accelerator\n",
    "!git pull"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 3 — Compile with CMake"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "!mkdir -p build && cd build && cmake .. -DCMAKE_BUILD_TYPE=Release && make -j4"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 4 — Run correctness tests"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "!python3 python/test_gpu_ops.py"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 5 — Run benchmark"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "!mkdir -p plots && ./build/benchmark | tee plots/benchmark_results.txt"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Step 6 — Generate performance charts"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "!python3 python/benchmark_viz.py\n",
    "\n",
    "from IPython.display import Image\n",
    "Image('plots/performance.png')"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## Results summary"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Parse and display results as a clean table\n",
    "import re\n",
    "\n",
    "with open('plots/benchmark_results.txt') as f:\n",
    "    content = f.read()\n",
    "\n",
    "print(content)\n",
    "\n",
    "# Extract the headline matmul speedup\n",
    "matches = re.findall(r'2048\\s+[\\d.]+\\s+[\\d.]+\\s+([\\d.]+)', content)\n",
    "if matches:\n",
    "    print(f'\\n>>> Headline result: {matches[0]}x speedup on 2048x2048 matrix multiply')"
   ]
  }
 ]
}